In [1]:
# ============================================================================
# TriSQL training script
# ============================================================================

from __future__ import annotations

import gc
import os
import re
from dataclasses import dataclass
from typing import Dict, List, Tuple, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, IterableDataset
from tqdm import tqdm

from datasets import load_dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup,
)

import sqlglot
from sqlglot import exp
from peft import LoraConfig, get_peft_model, TaskType

from sklearn.model_selection import train_test_split

import random
RANDOM_STATE = 42
random.seed(RANDOM_STATE) 

In [2]:
# ============================================================
# Configuration
# ============================================================

@dataclass
class Config:
    out_dir: str = "../trisql_models"
    bert_name: str = "bert-base-uncased"
    t5_name: str = "google/flan-t5-small"

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size: int = 2
    grad_accum: int = 2

    selector_epochs: int = 1
    generator_epochs: int = 5
    classifier_epochs: int = 1

    max_input_len: int = 512
    max_target_len: int = 256

    selector_lr: float = 2e-5
    classifier_lr: float = 2e-5
    gen_lr: float = 5e-5

    focal_gamma: float = 2.0
    focal_alpha: float = 0.25

    lambda_coef: float = 0.8
    tau: float = 0.6

    # dual-objective weights for generator
    lambda_struct: float = 0.5
    lambda_sql: float = 0.5

    seed: int = 42


CFG = Config()
DEVICE = torch.device(CFG.device)

for sub in ("selector", "generator", "classifier"):
    os.makedirs(os.path.join(CFG.out_dir, sub), exist_ok=True)

In [3]:
# ============================================================
# Utilities (unchanged from your original)
# ============================================================

def set_seed(seed: int = 42) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def parse_schema(schema_str: str) -> Dict[str, List[str]]:
    schema: Dict[str, List[str]] = {}
    for block in schema_str.split(" | "):
        if " : " not in block:
            continue
        table, cols_str = block.split(" : ", 1)
        table = table.strip()
        cols = []
        for item in cols_str.split(" , "):
            col = item.split(" (", 1)[0].strip()
            if col:
                cols.append(col)
        if table:
            schema[table] = cols
    return schema

def format_schema(schema: Dict[str, List[str]]) -> str:
    return "\n".join([f"Table {t}: {', '.join(cols)}" for t, cols in schema.items()])

def build_schema_text(schema: Dict[str, List[str]]) -> str:
    parts = []
    for table_name, columns in schema.items():
        col_str = ", ".join(columns)
        parts.append(f"{table_name}({col_str})")
    return " | ".join(parts)

def _safe_parse(sql: str):
    try:
        return sqlglot.parse_one(sql)
    except Exception:
        return None

def get_used_tables(sql: str, table_names: List[str]) -> Set[str]:
    parsed = _safe_parse(sql)
    if parsed is None:
        return set()
    tables = {t.name.lower() for t in parsed.find_all(exp.Table)}
    return {t for t in table_names if t.lower() in tables}

def get_used_columns(sql: str, schema: Dict[str, List[str]]) -> Set[Tuple[str, str]]:
    parsed = _safe_parse(sql)
    if parsed is None:
        return set()
    col_refs = set()
    for col in parsed.find_all(exp.Column):
        table = col.table.lower() if col.table else None
        name = col.name.lower()
        col_refs.add((table, name))
    used = set()
    for table, cols in schema.items():
        for col in cols:
            col_l = col.lower()
            if (table.lower(), col_l) in col_refs or (None, col_l) in col_refs:
                used.add((table, col))
    return used

def sql_complexity(sql: str) -> int:
    up = sql.upper()
    joins = len(re.findall(r"\bJOIN\b", up))
    nested_selects = len(re.findall(r"\bSELECT\b", up)) - 1
    has_group = bool(re.search(r"\bGROUP\s+BY\b", up))
    has_having = bool(re.search(r"\bHAVING\b", up))
    has_set_ops = bool(re.search(r"\b(UNION|INTERSECT|EXCEPT)\b", up))
    if nested_selects > 0 or has_set_ops or joins >= 3:
        return 2
    if joins >= 1 or has_group or has_having:
        return 1
    return 0

def clean_sql(sql: str) -> str:
    return re.sub(r"\s+", " ", sql).strip()

def extract_skeleton(sql: str) -> str:
    up = clean_sql(sql).upper()
    has_where = " WHERE " in up
    has_group = " GROUP BY " in up
    has_having = " HAVING " in up
    has_order = " ORDER BY " in up
    has_limit = " LIMIT " in up
    has_join = " JOIN " in up
    parts = ["SELECT [SELECT_FIELDS]", "FROM [TABLES]"]
    if has_join:
        parts.append("[JOIN_COND]")
    if has_where:
        parts.append("WHERE [WHERE_COND]")
    if has_group:
        parts.append("GROUP BY [GROUP_FIELDS]")
    if has_having:
        parts.append("HAVING [HAVING_COND]")
    if has_order:
        parts.append("ORDER BY [ORDER_FIELDS]")
    if has_limit:
        parts.append("LIMIT [LIMIT_VAL]")
    return " ".join(parts)



In [4]:
# ============================================================
# Spider loading
# ============================================================

def load_spider():
    spider = load_dataset("xlangai/spider")
    schemas_ds = load_dataset("richardr1126/spider-schema", split="train")
    schema_map = {
        row["db_id"]: parse_schema(row["Schema (values (type))"])
        for row in schemas_ds
    }
    def convert(split_name):
        records = []
        for item in spider[split_name]:
            db_id = item["db_id"]
            if db_id not in schema_map:
                continue
            records.append({
                "question": item["question"],
                "sql": clean_sql(item["query"]),   # key is 'sql'
                "db_id": db_id,
                "schema": schema_map[db_id],
            })
        return records

    train_data = convert("train")
    val_data = convert("validation")
    print(f"Loaded Spider: {len(train_data)} train, {len(val_data)} val")
    return train_data, val_data

In [5]:
# ============================================================
# 1) Question-Guided Schema Selector
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, labels):
        logits = logits.float().view(-1)
        labels = labels.float().view(-1)
        bce = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(labels == 1, probs, 1 - probs)
        alpha_t = torch.where(labels == 1,
                              torch.full_like(labels, self.alpha),
                              torch.full_like(labels, 1 - self.alpha))
        return (alpha_t * (1 - pt) ** self.gamma * bce).mean()

class QuestionGuidedSchemaSelector(nn.Module):
    def __init__(self, bert, tokenizer, device, hidden_size, lambda_coef=0.8):
        super().__init__()
        self.encoder = bert
        self.tokenizer = tokenizer
        self.device = device
        self.lambda_coef = lambda_coef
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.s_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def _encode(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(self.device)
        out = self.encoder(**inputs)
        return out.last_hidden_state[:, 0, :].squeeze(0)

    def forward(self, question, schema):
        tables = list(schema.keys())
        q_emb = self._encode(question)
        q_vec = self.q_proj(q_emb)

        t_embs = torch.stack([self._encode(t) for t in tables]) if tables else torch.empty(0, q_vec.size(-1), device=self.device)
        t_vecs = self.s_proj(t_embs) if tables else t_embs
        table_logits = (t_vecs @ q_vec) / (q_vec.size(-1) ** 0.5) if tables else torch.empty(0, device=self.device)

        col_logits_list = []
        for t in tables:
            cols = schema[t]
            if not cols:
                col_logits_list.append(torch.empty(0, device=self.device))
                continue
            c_embs = torch.stack([self._encode(c) for c in cols])
            c_vecs = self.s_proj(c_embs)
            c_logits = (c_vecs @ q_vec) / (q_vec.size(-1) ** 0.5)
            col_logits_list.append(c_logits)

        return table_logits, col_logits_list

def train_selector(train_data, val_data):
    print("\n=== Training schema selector ===")
    bert = AutoModel.from_pretrained(CFG.bert_name).to(DEVICE)
    tok = AutoTokenizer.from_pretrained(CFG.bert_name)

    model = QuestionGuidedSchemaSelector(
        bert=bert, tokenizer=tok, device=DEVICE,
        hidden_size=bert.config.hidden_size,
        lambda_coef=CFG.lambda_coef
    ).to(DEVICE)

    criterion = FocalLoss(CFG.focal_gamma, CFG.focal_alpha)
    optimizer = AdamW(model.parameters(), lr=CFG.selector_lr)

    steps_per_epoch = max(1, len(train_data) // CFG.grad_accum)
    total_steps = steps_per_epoch * CFG.selector_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, total_steps // 10), num_training_steps=total_steps
    )

    best_val = float("inf")

    for epoch in range(CFG.selector_epochs):
        model.train()
        optimizer.zero_grad()
        running = 0.0

        for i, ex in enumerate(tqdm(train_data, desc=f"selector epoch {epoch+1}")):
            q = ex["question"]
            sql = ex["sql"]
            schema = ex["schema"]
            tables = list(schema.keys())

            used_tables = get_used_tables(sql, tables)
            used_cols = get_used_columns(sql, schema)

            t_logits, c_logits_list = model(q, schema)
            loss = torch.tensor(0.0, device=DEVICE)

            if len(tables) > 0:
                t_labels = torch.tensor([1.0 if t in used_tables else 0.0 for t in tables], device=DEVICE)
                loss = loss + criterion(t_logits, t_labels)

                col_logits_all, col_labels_all = [], []
                for idx, t in enumerate(tables):
                    cols = schema[t]
                    if not cols:
                        continue
                    c_logits = c_logits_list[idx]
                    c_labels = torch.tensor([1.0 if (t, c) in used_cols else 0.0 for c in cols], device=DEVICE)
                    if c_logits.numel() == c_labels.numel() and c_logits.numel() > 0:
                        col_logits_all.append(c_logits)
                        col_labels_all.append(c_labels)

                if col_logits_all:
                    loss = loss + criterion(torch.cat(col_logits_all), torch.cat(col_labels_all))

            (loss / CFG.grad_accum).backward()
            running += loss.item()

            if (i + 1) % CFG.grad_accum == 0 or (i + 1) == len(train_data):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            # validation every 200 steps
            if i > 0 and i % 200 == 0:
                model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for exv in val_data:
                        qv = exv["question"]
                        sqlv = exv["sql"]
                        schemav = exv["schema"]
                        tablesv = list(schemav.keys())
                        used_tables_v = get_used_tables(sqlv, tablesv)
                        used_cols_v = get_used_columns(sqlv, schemav)

                        t_logits_v, c_logits_list_v = model(qv, schemav)
                        lv = torch.tensor(0.0, device=DEVICE)
                        if len(tablesv) > 0:
                            t_labels_v = torch.tensor([1.0 if t in used_tables_v else 0.0 for t in tablesv], device=DEVICE)
                            lv = lv + criterion(t_logits_v, t_labels_v)
                            col_logits_all_v, col_labels_all_v = [], []
                            for idx, t in enumerate(tablesv):
                                cols = schemav[t]
                                if not cols:
                                    continue
                                c_logits_v = c_logits_list_v[idx]
                                c_labels_v = torch.tensor([1.0 if (t, c) in used_cols_v else 0.0 for c in cols], device=DEVICE)
                                if c_logits_v.numel() == c_labels_v.numel() and c_logits_v.numel() > 0:
                                    col_logits_all_v.append(c_logits_v)
                                    col_labels_all_v.append(c_labels_v)
                            if col_logits_all_v:
                                lv = lv + criterion(torch.cat(col_logits_all_v), torch.cat(col_labels_all_v))
                        val_loss += lv.item()
                avg_val = val_loss / max(1, len(val_data))
                print(f"train_loss={running/(i+1):.4f}  val_loss={avg_val:.4f}")
                if avg_val < best_val:
                    best_val = avg_val
                    torch.save(model.state_dict(), os.path.join(CFG.out_dir, "selector", "model.pt"))
                    bert.save_pretrained(os.path.join(CFG.out_dir, "selector", "bert"))
                    tok.save_pretrained(os.path.join(CFG.out_dir, "selector", "bert"))
                    print("  saved best selector")
                model.train()

    # final save if not saved yet
    if not os.path.exists(os.path.join(CFG.out_dir, "selector", "model.pt")):
        torch.save(model.state_dict(), os.path.join(CFG.out_dir, "selector", "model.pt"))
        bert.save_pretrained(os.path.join(CFG.out_dir, "selector", "bert"))
        tok.save_pretrained(os.path.join(CFG.out_dir, "selector", "bert"))

    del model, bert, tok
    gc.collect()
    torch.cuda.empty_cache()

In [6]:
# ============================================================
# 2) Structure-Aware SQL Generator (dual-objective)
# ============================================================

class SkeletonDataset(Dataset):
    """Predict the skeleton from question + schema."""
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        question = item["question"]
        schema_text = build_schema_text(item["schema"])
        skeleton = extract_skeleton(item["sql"])    # target skeleton

        source = f"question: {question} schema: {schema_text}"
        model_inputs = self.tokenizer(source, max_length=CFG.max_input_len, truncation=True, padding="max_length", return_tensors="pt")
        target = self.tokenizer(text_target=skeleton, max_length=CFG.max_target_len, truncation=True, padding="max_length", return_tensors="pt")
        labels = target["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels": labels,
        }

class SQLDataset(Dataset):
    """Predict the full SQL from question + skeleton + schema."""
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        question = item["question"]
        schema_text = build_schema_text(item["schema"])
        skeleton = extract_skeleton(item["sql"])
        sql = item["sql"]          # target full SQL

        source = f"question: {question} skeleton: {skeleton} schema: {schema_text}"
        model_inputs = self.tokenizer(source, max_length=CFG.max_input_len, truncation=True, padding="max_length", return_tensors="pt")
        target = self.tokenizer(text_target=sql, max_length=CFG.max_target_len, truncation=True, padding="max_length", return_tensors="pt")
        labels = target["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels": labels,
        }

def train_generator(train_data, val_data):
    print("\n=== Training structure-aware generator (skeleton + SQL) ===")
    tokenizer = AutoTokenizer.from_pretrained(CFG.t5_name)
    model = T5ForConditionalGeneration.from_pretrained(
        CFG.t5_name
    ).to(DEVICE)

    # create datasets
    train_skel = SkeletonDataset(train_data, tokenizer)
    train_sql = SQLDataset(train_data, tokenizer)
    val_skel = SkeletonDataset(val_data, tokenizer)
    val_sql = SQLDataset(val_data, tokenizer)

    # train DataLoaders
    skel_loader = DataLoader(train_skel, batch_size=CFG.batch_size, shuffle=True)
    sql_loader  = DataLoader(train_sql,  batch_size=CFG.batch_size, shuffle=True)

    # validation DataLoaders (FIX: wrap datasets in DataLoader)
    val_skel_loader = DataLoader(val_skel, batch_size=CFG.batch_size, shuffle=False)
    val_sql_loader  = DataLoader(val_sql,  batch_size=CFG.batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=CFG.gen_lr)
    total_steps = (len(skel_loader) + len(sql_loader)) * CFG.generator_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps,
    )

    best_val_loss = float("inf")

    for epoch in range(CFG.generator_epochs):
        model.train()
        optimizer.zero_grad()
        running_loss = 0.0
        skel_iter = iter(skel_loader)
        sql_iter = iter(sql_loader)
        max_batches = max(len(skel_loader), len(sql_loader))

        for step in tqdm(range(max_batches), desc=f"gen epoch {epoch+1}"):
            # fetch skeleton batch (if available)
            try:
                skel_batch = next(skel_iter)
            except StopIteration:
                skel_batch = None

            # fetch SQL batch
            try:
                sql_batch = next(sql_iter)
            except StopIteration:
                sql_batch = None

            loss = torch.tensor(0.0, device=DEVICE)
            if skel_batch is not None:
                skel_batch = {k: v.to(DEVICE) for k, v in skel_batch.items()}
                out = model(**skel_batch)
                loss = loss + CFG.lambda_struct * out.loss

            if sql_batch is not None:
                sql_batch = {k: v.to(DEVICE) for k, v in sql_batch.items()}
                out = model(**sql_batch)
                loss = loss + CFG.lambda_sql * out.loss

            if loss.item() > 0:
                loss.backward()
                running_loss += loss.item()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            if step % 500 == 0 and step>0:
                # ---- validation using DataLoaders (fixed) ----
                model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for batch in val_skel_loader:          # <- now a DataLoader
                        batch = {k: v.to(DEVICE) for k, v in batch.items()}
                        val_loss += model(**batch).loss.item()
                    for batch in val_sql_loader:           # <- now a DataLoader
                        batch = {k: v.to(DEVICE) for k, v in batch.items()}
                        val_loss += model(**batch).loss.item()

                avg_val = val_loss / (len(val_skel_loader) + len(val_sql_loader))
                avg_train = running_loss / max(1, max_batches)
                print(f"train_loss={avg_train:.4f}  val_loss={avg_val:.4f}")

                if avg_val < best_val_loss:
                    best_val_loss = avg_val
                    model.save_pretrained(os.path.join(CFG.out_dir, "generator", "model"))
                    tokenizer.save_pretrained(os.path.join(CFG.out_dir, "generator", "tokenizer"))
                    print("  saved best generator")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [7]:
# ============================================================
# 3) Complexity-Aware SQL Classifier
# ============================================================

class ComplexityClassifier(nn.Module):
    def __init__(self, encoder, hidden_size, num_classes=3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(out.last_hidden_state[:, 0, :])

class ComplexityDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.examples = examples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        text = f"question: {ex['question']} sql: {ex['sql']} schema: {format_schema(ex['schema'])}"
        enc = self.tokenizer(text, truncation=True, padding="max_length",
                             max_length=CFG.max_input_len, return_tensors="pt")
        return {
            "input_ids": enc.input_ids.squeeze(0),
            "attention_mask": enc.attention_mask.squeeze(0),
            "labels": torch.tensor(sql_complexity(ex["sql"]), dtype=torch.long),
        }

def train_classifier(train_data, val_data):
    print("\n=== Training complexity classifier ===")
    tokenizer = AutoTokenizer.from_pretrained(CFG.bert_name)
    encoder = AutoModel.from_pretrained(CFG.bert_name).to(DEVICE)
    model = ComplexityClassifier(encoder, encoder.config.hidden_size).to(DEVICE)

    train_ds = ComplexityDataset(train_data, tokenizer)
    val_ds = ComplexityDataset(val_data, tokenizer)

    optimizer = AdamW(model.parameters(), lr=CFG.classifier_lr)
    criterion = nn.CrossEntropyLoss()

    steps_per_epoch = max(1, len(train_ds) // CFG.batch_size)
    total_steps = steps_per_epoch * CFG.classifier_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(1, total_steps//10), num_training_steps=total_steps)

    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False)

    best_val = float("inf")

    for epoch in range(CFG.classifier_epochs):
        model.train()
        running = 0.0
        for batch in tqdm(train_loader, desc=f"classifier epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            running += loss.item()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)
                logits = model(input_ids, attention_mask)
                val_loss += criterion(logits, labels).item()

        avg_val = val_loss / max(1, len(val_loader))
        avg_train = running / max(1, len(train_loader))
        print(f"train_loss={avg_train:.4f}  val_loss={avg_val:.4f}")

        if avg_val < best_val:
            best_val = avg_val
            torch.save(model.state_dict(), os.path.join(CFG.out_dir, "classifier", "model.pt"))
            encoder.save_pretrained(os.path.join(CFG.out_dir, "classifier", "bert"))
            tokenizer.save_pretrained(os.path.join(CFG.out_dir, "classifier", "bert"))
            print("  saved best classifier")

    del model, encoder, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [8]:
import random

def parse_create_tables(input_text: str) -> dict:
    """Parse CREATE TABLE statements into {table_name: [col1, col2, ...]} format."""
    schema = {}
    # Non-greedy match to handle multiple concatenated CREATE TABLE statements
    pattern = re.compile(r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?(\w+)\s*\((.*?)\)', re.DOTALL | re.IGNORECASE)
    
    for match in pattern.finditer(input_text):
        table_name = match.group(1)
        cols_str = match.group(2)
        cols = []
        for col_def in cols_str.split(','):
            col_def = col_def.strip()
            if not col_def:
                continue
            # Column name is usually the first token before type/constraints
            col_name = col_def.split()[0].strip().strip('"').strip('`').strip("'")
            # Filter out SQL constraint keywords that might appear as first tokens
            if col_name and col_name.upper() not in ('PRIMARY', 'KEY', 'CONSTRAINT', 
                                                     'FOREIGN', 'REFERENCES', 'CHECK', 
                                                     'UNIQUE', 'INDEX', 'LIKE'):
                cols.append(col_name)
        if cols:
            schema[table_name] = cols
    return schema

def load_and_merge_custom_dataset(spider_train, spider_val, custom_ds_name="DipamSoni/custom_text_to_sql_dataset", train_ratio=0.8, seed=42):
    print(f"\n=== Loading & Merging {custom_ds_name} ===")
    ds = load_dataset(custom_ds_name)
    
    # Handle dataset structure (usually 'train' split exists)
    if 'train' in ds:
        data_iter = ds['train']
    elif len(ds) == 1:
        data_iter = ds[list(ds.keys())[0]]
    else:
        data_iter = []
        for split in ds.values():
            data_iter.extend(split)

    converted = []
    for item in data_iter:
        question = item.get("instruction", "").strip()
        sql = clean_sql(item.get("response", ""))
        schema_raw = item.get("input", "")
        
        if not question or not sql:
            continue
            
        schema = parse_create_tables(schema_raw)
        if not schema:
            continue
            
        converted.append({
            "question": question,
            "sql": sql,
            "db_id": "custom_dataset",
            "schema": schema
        })

    print(f"Parsed {len(converted)} valid samples from custom dataset.")

    # 80/20 split
    random.seed(seed)
    random.shuffle(converted)
    split_idx = int(len(converted) * train_ratio)
    custom_train = converted[:split_idx]
    custom_val = converted[split_idx:]

    # Merge with Spider
    new_train = spider_train + custom_train
    new_val = spider_val + custom_val

    print(f"Dataset merged successfully:")
    print(f"  Train: {len(spider_train)} (Spider) + {len(custom_train)} (Custom) = {len(new_train)}")
    print(f"  Val:   {len(spider_val)} (Spider) + {len(custom_val)} (Custom) = {len(new_val)}")
    
    # Optional: sanity check
    print("\nSample merged record:")
    print(new_train[-1])
    
    return new_train, new_val

In [9]:
# ============================================================
# Main
# ============================================================

set_seed(CFG.seed)
train_data, val_data = load_spider()
spider_train, spider_val = load_spider()
train_data, val_data = load_and_merge_custom_dataset(spider_train, spider_val, train_ratio=0.8, seed=CFG.seed)

# Taking a subset of the data
random.shuffle(train_data)
val, test = train_test_split(val_data, test_size=0.998, random_state=RANDOM_STATE, shuffle=True)
print(len(train_data))
print(len(val))
print(len(test))

Loaded Spider: 7000 train, 1034 val
Loaded Spider: 7000 train, 1034 val

=== Loading & Merging DipamSoni/custom_text_to_sql_dataset ===
Parsed 366314 valid samples from custom dataset.
Dataset merged successfully:
  Train: 7000 (Spider) + 293051 (Custom) = 300051
  Val:   1034 (Spider) + 73263 (Custom) = 74297

Sample merged record:
{'question': 'What was the Attendance on May 12, when the New York Yankees were the Opponent?', 'sql': 'SELECT attendance FROM table_name_71 WHERE opponent = "new york yankees" AND date = "may 12"', 'db_id': 'custom_dataset', 'schema': {'table_name_71': ['attendance', 'opponent', 'date']}}
300051
148
74149


In [17]:
train_selector(train_data[:10000], val)


=== Training schema selector ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
selector epoch 1:   2%|▏         | 200/10000 [01:45<1:01:25,  2.66it/s]

train_loss=0.1526  val_loss=0.1342


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

train_loss=0.0772  val_loss=0.0577


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

selector epoch 1:  98%|█████████▊| 9801/10000 [1:23:56<13:32,  4.08s/it]

  saved best selector


selector epoch 1: 100%|██████████| 10000/10000 [1:25:35<00:00,  1.95it/s]


In [13]:
train_generator(train_data[:10000], val)


=== Training structure-aware generator (skeleton + SQL) ===


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
gen epoch 1:  10%|█         | 500/5000 [02:02<18:22,  4.08it/s]

train_loss=0.2880  val_loss=1.3680


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

gen epoch 5:  80%|████████  | 4002/5000 [16:34<22:10,  1.33s/it]

train_loss=0.1017  val_loss=0.2254


gen epoch 5:  90%|█████████ | 4501/5000 [18:36<15:12,  1.83s/it]

train_loss=0.1119  val_loss=0.2258


gen epoch 5: 100%|██████████| 5000/5000 [20:34<00:00,  4.05it/s]


In [ ]:
generator_tokenizer = AutoTokenizer.from_pretrained(
            "./trisql_models/generator/tokenizer/"
            )
generator = T5ForConditionalGeneration.from_pretrained(
            "./trisql_models/generator/model/"
        ).to('cuda')

In [ ]:
train_classifier(train_data, val_data)

In [12]:
print("question", val_data[10]['question'])
print("\n")
print("schema:", format_schema(val_data[10]['schema']))
print("\n")
print("sql:", val_data[10]['sql'])
print("\n")
print("used columns:", get_used_columns(val_data[10]['sql'], val_data[10]['schema']))
print("\n")
print("used_tables:", get_used_tables(val_data[10]['sql'], val_data[10]['schema']))

question Show all countries and the number of singers in each country.


schema: Table stadium: Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average
Table singer: Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male
Table concert: concert_ID, concert_Name, Theme, Stadium_ID, Year
Table singer_in_concert: concert_ID, Singer_ID


sql: SELECT country , count(*) FROM singer GROUP BY country


used columns: {('singer', 'Country')}


used_tables: {'singer'}


In [ ]:
source = f"question: {val_data[0]['question']} schema: {format_schema(val_data[0]['schema'])}"
inputs = generator_tokenizer(
    source, return_tensors="pt", truncation=True, max_length=512
).to('cuda')
with torch.no_grad():
    outputs = generator.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=1,
        do_sample=False,
    )
skeleton =generator_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(skeleton)

source = f"question: {val_data[0]['question']} skeleton: {skeleton} schema: {val_data[0]['schema']}"
inputs = generator_tokenizer(
    source, return_tensors="pt", truncation=True, max_length=512
).to('cuda')
with torch.no_grad():
    outputs = generator.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=1,
        do_sample=False,
    )
sql = generator_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(sql)

### Metrics calculation

In [18]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_selector(model, data):
    model.eval()
    table_preds, table_trues = [], []
    col_preds, col_trues = [], []
    
    with torch.no_grad():
        for ex in tqdm(data, desc="eval selector"):
            q = ex["question"]
            sql = ex["sql"]
            schema = ex["schema"]
            tables = list(schema.keys())
            
            used_tables = get_used_tables(sql, tables)
            used_cols = get_used_columns(sql, schema)
            
            t_logits, c_logits_list = model(q, schema)
            
            # Table predictions (prob > 0.5 after sigmoid)
            t_probs = torch.sigmoid(t_logits).cpu().numpy()
            t_pred = (t_probs > 0.5).astype(int)
            t_true = np.array([1.0 if t in used_tables else 0.0 for t in tables])
            table_preds.extend(t_pred)
            table_trues.extend(t_true)
            
            for idx, t in enumerate(tables):
                cols = schema[t]
                if not cols:
                    continue
                c_probs = torch.sigmoid(c_logits_list[idx]).cpu().numpy()
                c_pred = (c_probs > 0.5).astype(int)
                c_true = np.array([1.0 if (t, c) in used_cols else 0.0 for c in cols])
                col_preds.extend(c_pred)
                col_trues.extend(c_true)
                
    # Overall table-level metrics
    table_prec = precision_score(table_trues, table_preds, zero_division=0)
    table_rec = recall_score(table_trues, table_preds, zero_division=0)
    table_f1 = f1_score(table_trues, table_preds, zero_division=0)
    
    # Overall column-level metrics
    col_prec = precision_score(col_trues, col_preds, zero_division=0)
    col_rec = recall_score(col_trues, col_preds, zero_division=0)
    col_f1 = f1_score(col_trues, col_preds, zero_division=0)
    
    print(f"Selector Table  -> P: {table_prec:.4f}  R: {table_rec:.4f}  F1: {table_f1:.4f}")
    print(f"Selector Column -> P: {col_prec:.4f}  R: {col_rec:.4f}  F1: {col_f1:.4f}")
    return {"table_prec": table_prec, "table_rec": table_rec, "table_f1": table_f1,
            "col_prec": col_prec, "col_rec": col_rec, "col_f1": col_f1}

# Usage after training:
selector_model = QuestionGuidedSchemaSelector(
    bert=AutoModel.from_pretrained(os.path.join(CFG.out_dir, "selector", "bert")).to(DEVICE),
    tokenizer=AutoTokenizer.from_pretrained(os.path.join(CFG.out_dir, "selector", "bert")),
    device=DEVICE,
    hidden_size=AutoModel.from_pretrained(os.path.join(CFG.out_dir, "selector", "bert")).config.hidden_size,
    lambda_coef=CFG.lambda_coef
).to(DEVICE)
selector_model.load_state_dict(torch.load(os.path.join(CFG.out_dir, "selector", "model.pt")))
metrics = evaluate_selector(selector_model, val)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

eval selector: 100%|██████████| 148/148 [00:12<00:00, 11.76it/s]

Selector Table  -> P: 0.9706  R: 0.6600  F1: 0.7857
Selector Column -> P: 0.8163  R: 0.2030  F1: 0.3252


In [14]:
def normalize_sql(sql):
    # Basic normalization: lowercase, collapse whitespace, remove trailing semicolon
    sql = sql.strip().lower().rstrip(";")
    sql = " ".join(sql.split())
    return sql

def evaluate_generator(generator, tokenizer, data):
    generator.eval()
    skel_em = 0
    sql_em = 0
    total = len(data)
    
    for ex in tqdm(data, desc="eval generator"):
        question = ex["question"]
        schema = ex["schema"]
        ref_sql = normalize_sql(clean_sql(ex["sql"]))
        ref_skeleton = extract_skeleton(ex["sql"])  # already normalized
        
        schema_text = build_schema_text(schema)
        
        # Generate skeleton
        source_skel = f"question: {question} schema: {schema_text}"
        inputs_skel = tokenizer(source_skel, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            outputs_skel = generator.generate(**inputs_skel, max_new_tokens=128, num_beams=1, do_sample=False)
        pred_skeleton = tokenizer.decode(outputs_skel[0], skip_special_tokens=True).strip()
        
        if normalize_sql(pred_skeleton) == normalize_sql(ref_skeleton):
            skel_em += 1
        
        # Generate SQL
        source_sql = f"question: {question} skeleton: {pred_skeleton} schema: {schema_text}"
        inputs_sql = tokenizer(source_sql, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            outputs_sql = generator.generate(**inputs_sql, max_new_tokens=256, num_beams=1, do_sample=False)
        pred_sql = tokenizer.decode(outputs_sql[0], skip_special_tokens=True)
        pred_sql_norm = normalize_sql(clean_sql(pred_sql))
        
        if pred_sql_norm == ref_sql:
            sql_em += 1
    
    skel_em = skel_em / total
    sql_em = sql_em / total
    print(f"Generator Skeleton EM: {skel_em:.4f}")
    print(f"Generator SQL EM     : {sql_em:.4f}")
    return {"skel_em": skel_em, "sql_em": sql_em}

# Usage:
generator_tokenizer = AutoTokenizer.from_pretrained(os.path.join(CFG.out_dir, "generator", "tokenizer"))
generator_model = T5ForConditionalGeneration.from_pretrained(os.path.join(CFG.out_dir, "generator", "model")).to(DEVICE)
generator_metrics = evaluate_generator(generator_model, generator_tokenizer, val)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
eval generator: 100%|██████████| 148/148 [01:14<00:00,  1.98it/s]

Generator Skeleton EM: 0.8446
Generator SQL EM     : 0.1824


In [16]:
def evaluate_classifier(classifier, tokenizer, data):
    classifier.eval()
    correct = 0
    total = 0
    ds = ComplexityDataset(data, tokenizer)
    loader = DataLoader(ds, batch_size=CFG.batch_size, shuffle=False)
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="eval classifier"):
            input_ids = batch["input_ids"].to(DEVICE)
            attn = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = classifier(input_ids, attn)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"Classifier Accuracy: {acc:.4f}")
    return acc

# Usage:
classifier_tokenizer = AutoTokenizer.from_pretrained(os.path.join(CFG.out_dir, "classifier", "bert"))
classifier_encoder = AutoModel.from_pretrained(os.path.join(CFG.out_dir, "classifier", "bert")).to(DEVICE)
classifier_model = ComplexityClassifier(classifier_encoder, classifier_encoder.config.hidden_size).to(DEVICE)
classifier_model.load_state_dict(torch.load(os.path.join(CFG.out_dir, "classifier", "model.pt")))
classifier_acc = evaluate_classifier(classifier_model, classifier_tokenizer, val)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

eval classifier: 100%|██████████| 74/74 [00:03<00:00, 21.82it/s]

Classifier Accuracy: 0.9730


In [ ]:
from ../services/text_to_sql/app/model_inference import TriSQLInference

def evaluate_full_pipeline(inference_engine, data, device="cuda"):
    em = 0
    total = len(data)
    for ex in tqdm(data, desc="full pipeline"):
        question = ex["question"]
        schema = ex["schema"]
        ref_sql = normalize_sql(clean_sql(ex["sql"]))
        
        result = inference_engine.predict(question, schema)
        pred_sql = normalize_sql(clean_sql(result["sql"]))
        if pred_sql == ref_sql:
            em += 1
    em = em / total
    print(f"Full Pipeline EM: {em:.4f}")
    return em

# Usage after training all components:
# engine = TriSQLInference(model_dir=CFG.out_dir)
# evaluate_full_pipeline(engine, test)   # test is your 74k split

In [ ]:
# ------- After selector training -------
selector_bert = AutoModel.from_pretrained(os.path.join(CFG.out_dir, "selector", "bert")).to(DEVICE)
selector_tok = AutoTokenizer.from_pretrained(os.path.join(CFG.out_dir, "selector", "bert"))
selector = QuestionGuidedSchemaSelector(
    bert=selector_bert, tokenizer=selector_tok, device=DEVICE,
    hidden_size=selector_bert.config.hidden_size, lambda_coef=CFG.lambda_coef
).to(DEVICE)
selector.load_state_dict(torch.load(os.path.join(CFG.out_dir, "selector", "model.pt")))
selector.eval()
evaluate_selector(selector, val)   # or test

# ------- After generator training -------
gen_tok = AutoTokenizer.from_pretrained(os.path.join(CFG.out_dir, "generator", "tokenizer"))
gen = T5ForConditionalGeneration.from_pretrained(os.path.join(CFG.out_dir, "generator", "model")).to(DEVICE)
gen.eval()
evaluate_generator(gen, gen_tok, val)   # or test

# ------- After classifier training -------
classifier_bert = AutoModel.from_pretrained(os.path.join(CFG.out_dir, "classifier", "bert")).to(DEVICE)
classifier_tok = AutoTokenizer.from_pretrained(os.path.join(CFG.out_dir, "classifier", "bert"))
classifier = ComplexityClassifier(classifier_bert, classifier_bert.config.hidden_size).to(DEVICE)
classifier.load_state_dict(torch.load(os.path.join(CFG.out_dir, "classifier", "model.pt")))
classifier.eval()
evaluate_classifier(classifier, classifier_tok, val)

# ------- Full pipeline -------
engine = TriSQLInference(model_dir=CFG.out_dir)
evaluate_full_pipeline(engine, test)   # use your 74k test split